# Economic indicators

World Bank panel since 2000. The consolidated charts and tables live in **`report.html`** (regenerate with `render_report.py`).

This notebook loads the cache, prints summary stats, and fits the pooled regression on the six-country comparison set.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import HTML, display

ROOT = Path.cwd()
if ROOT.name != "economic-data":
    candidate = ROOT / "experiments" / "economic-data"
    if candidate.exists():
        ROOT = candidate

sys.path.insert(0, str(ROOT))
from fetch_world_bank import CACHE_PATH, load_or_fetch
from panels import INDICATORS, to_wide
from render_report import build_report

long = load_or_fetch(CACHE_PATH)
wide = to_wide(long)
focus = wide[wide["is_focus"]].copy()
countries = wide[~wide["is_aggregate"]].copy()

print(
    f"{countries['iso'].nunique()} countries, "
    f"years {int(wide['year'].min())}–{int(wide['year'].max())}"
)
display(focus[INDICATORS].describe().round(2))

In [ ]:
print("Pooled correlations (six-country panel):")
display(focus[INDICATORS].corr().round(3))

print("\nWithin-country correlations:")
display(
    focus.groupby("country")[INDICATORS]
    .corr()
    .unstack()
    .loc[:, [("inflation", "unemployment"), ("gdp_growth", "inflation")]]
    .round(3)
)

In [ ]:
model_df = focus.dropna(subset=["inflation", "unemployment"])
model = smf.ols("inflation ~ unemployment + C(country)", data=model_df).fit()
print(model.summary())

In [ ]:
report_path = build_report(long)
display(HTML(f'<p>Wrote <a href="{report_path.name}" target="_blank">{report_path.name}</a> — open it for charts and tables.</p>'))